# E1 — Final Held-Out Test Evaluation (Prostate158, 19 cases)

**Thesis:** Segmentasi Zona Anatomi pada Citra MRI Pasien Kanker Prostat
menggunakan 2D U-Net dan Rekonstruksi 3D.

**RQ1:** *Berapa Hasil Evaluasi dari Segmentasi Citra MRI pada Pasien Kanker
Prostat menggunakan 2D U-Net?*

---

## 1. Purpose

This notebook evaluates the **selected validation candidate** — experiment
`exp_e1_dice_ce`, epoch 88 — on the official Prostate158 held-out test set.
It performs inference only: no training, no fine-tuning, no model selection,
no threshold tuning, no post-processing, no test-time augmentation.

### Status of E1

E1 is the **selected validation candidate**. It is *not yet* the final model —
that designation is only earned once this held-out evaluation has actually run.
Until then, nothing in this notebook refers to E1 as final.

E1 was chosen on **validation data only** (20 cases), by the pre-committed rule
*highest validation macro Dice*:

| Arm | Intervention | Best epoch | CG Dice | PZ Dice | Macro Dice |
|---|---|---|---|---|---|
| Baseline | Cross Entropy | 77 | 0.8642 | 0.7375 | 0.8008 |
| **E1 ← selected** | **+ SoftDice(CG, PZ)** | **88** | **0.8662** | **0.7527** | **0.8094** |
| E2 | + augmentation | 70 | 0.8636 | 0.7327 | 0.7982 |
| E3 | + CE class weights | 99 | 0.8652 | 0.7510 | 0.8081 |

> All four configurations performed within a relatively narrow validation range
> (macro Dice spread 0.0112, smaller than any single arm's per-case SD of ≈0.05–0.07).
> E1 was selected because it achieved the highest validation macro Dice, CG Dice
> and PZ Dice, while also showing the strongest PZ ASD and competitive PZ HD95.
> E1 is **not** claimed to be statistically superior: the one available paired
> comparison (E3 vs E1, n = 20) gives p = 0.330 with a bootstrap CI of
> [−0.0091, +0.0073].

### The rule this notebook exists to protect

**Whatever numbers come out of section 11, they do not change the selected model.**
If E1 scores lower on test than on validation, that is the finding and it is
reported as such. Re-running with a different arm after seeing these numbers would
turn the held-out set into a second validation set.

### Historical note on the test set

> **Historical note:** the official Prostate158 test set has previously been
> evaluated using the baseline experiment. Therefore, this E1 evaluation is **not**
> the first historical use of the official test set.

Prior uses, both with the **Baseline epoch-77** checkpoint and both **before** the
E1/E2/E3 tuning arms were designed:

| Date | What | Checkpoint |
|---|---|---|
| 2026-09-13 | baseline test evaluation (`results/exp01_baseline/eval_test_*`) | Baseline ep. 77 |
| 2026-09-19 | RQ2 reconstruction geometry study (`results/exp04_reconstruction/`) | Baseline ep. 77 |
| this run | first test evaluation of **E1** | E1 ep. 88 |

RQ1 model selection was nonetheless made on validation only: every arm's selection
artifact records `split: val`, `n_cases: 20`, `is_official_held_out_test: false`,
and no arm's training config declares `test_csv` / `test_dir`. Those historical
results are left untouched by this notebook.

### Label mapping

`0 = Background`, `1 = Central Gland (CG = CZ + TZ)`, `2 = Peripheral Zone (PZ)` —
verified 2026-10-05 by multi-planar inspection of cases 020, 059, 099 and 139
(`results/label_mapping/label_mapping_verified.md`). The mapping is **not** changed
anywhere in this notebook.

### Scope

This notebook covers **RQ1 only**. No 3D reconstruction, mesh generation, surface
reconstruction or RQ2 evaluation is performed here — see section 16.

---

## How to run

1. `Runtime > Change runtime type > T4 GPU`
2. **Run all.** With `ONE_TIME_RUN_CONFIRMED = False` (the default) every check runs
   and the notebook stops at section 9, before touching the test set.
3. Review the checks, set `ONE_TIME_RUN_CONFIRMED = True` in section 2, **Run all**
   again to perform the evaluation.

Re-running after a completed evaluation is safe: section 8 detects it and reuses the
existing results instead of evaluating the held-out set twice.


## 2. Configuration

Everything adjustable lives here. `ONE_TIME_RUN_CONFIRMED` is the only switch you
should normally change; the paths only need editing if your Drive layout differs.

Expected Drive layout:

```
THESIS_PROSTATE158/
├── dataset/
│   ├── prostate158_train/          119 train + 20 validation cases
│   └── test/
│       ├── prostate158_test.zip    official 19-case archive
│       └── extracted/
└── results/
    ├── exp01_baseline/             historical — not written to
    ├── exp_e1_dice_ce/             E1 training/validation — read-only here
    ├── exp_e2_ce_aug/              historical — not written to
    ├── exp_e3_dice_ce_pzweight/    historical — not written to
    └── test_e1_epoch88/            ← the ONLY directory this notebook writes to
```


In [ ]:
# ========================= THE ONLY SWITCH =========================
# False: run every check, print the test plan, then stop before inference.
# True : perform the held-out evaluation.  NEVER commit this as True.
ONE_TIME_RUN_CONFIRMED = False
# ===================================================================

# ---- Paths -------------------------------------------------------------
DRIVE_ROOT        = '/content/drive/MyDrive/THESIS_PROSTATE158'
PROJECT_ROOT      = '/content/thesis-project'          # repository clone (ephemeral)
REPO_URL          = 'https://github.com/amadeussandro/prostate-mri-segmentation-thesis.git'
REPO_BRANCH       = 'main'

CHECKPOINT_PATH   = f'{DRIVE_ROOT}/results/exp_e1_dice_ce/best_model.pt'
TEST_ARCHIVE_PATH = f'{DRIVE_ROOT}/dataset/test/prostate158_test.zip'
TEST_EXTRACT_DIR  = f'{DRIVE_ROOT}/dataset/test/extracted'

# Isolated output directory. The E1 experiment directory is never written to.
OUTPUT_DIR        = f'{DRIVE_ROOT}/results/test_e1_epoch88'

# ---- Frozen candidate: assertions, not settings ------------------------
EXPERIMENT_NAME = 'exp_e1_dice_ce'
EXPECTED_SHA256 = '2366e3c68f5ec616b19c0e7df39f9135898fd287b9436ce5dfab7baf8528d5d2'
EXPECTED_EPOCH  = 88
EXPECTED_LOSS   = 'dice_ce'      # Dice + Cross Entropy
EXPECTED_CE_CLASS_WEIGHTS = None # unweighted CE -- E1, not E3

N_TEST_CASES  = 19
N_TRAIN_CASES = 119
N_VAL_CASES   = 20
SEED = 42

LABEL_MAPPING = {0: 'Background', 1: 'CG', 2: 'PZ'}
CG, PZ = 1, 2

# E1 validation anchors, for the generalization-gap table in section 13.
E1_VALIDATION = {
    'cg_dice': 0.8662, 'pz_dice': 0.7527, 'macro_dice': 0.8094,
    'cg_precision': 0.8823, 'cg_recall': 0.8604,
    'pz_precision': 0.8073, 'pz_recall': 0.7161,
    'cg_hd95': 5.2467, 'pz_hd95': 5.4061, 'cg_asd': 1.1508, 'pz_asd': 0.9453,
}

print('E1 FINAL HELD-OUT TEST -- configuration')
print('-' * 70)
print(f'  confirmation   : {ONE_TIME_RUN_CONFIRMED}')
print(f'  checkpoint     : {CHECKPOINT_PATH}')
print(f'  expected SHA   : {EXPECTED_SHA256}')
print(f'  expected epoch : {EXPECTED_EPOCH}')
print(f'  test archive   : {TEST_ARCHIVE_PATH}')
print(f'  test extracted : {TEST_EXTRACT_DIR}')
print(f'  output dir     : {OUTPUT_DIR}')
print('-' * 70)
if not ONE_TIME_RUN_CONFIRMED:
    print('  Confirmation is OFF: the notebook will stop at section 9,')
    print('  before the held-out set is touched.')


## 3. Google Drive and repository setup

Drive holds the frozen checkpoint, the dataset and the results; the repository
supplies the evaluation code. Every artifact of record is written to Drive, never
to `/content`, which Colab wipes when the VM is recycled.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, sys

if not os.path.isdir(PROJECT_ROOT):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run(['git', 'checkout', REPO_BRANCH], check=True)
subprocess.run(['git', 'pull', 'origin', REPO_BRANCH], check=True)

GIT_COMMIT = subprocess.run(['git', '-C', PROJECT_ROOT, 'rev-parse', 'HEAD'],
                            capture_output=True, text=True).stdout.strip()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('Repository :', PROJECT_ROOT)
print('Branch     :', REPO_BRANCH)
print('Commit     :', GIT_COMMIT)
subprocess.run(['git', '-C', PROJECT_ROOT, 'log', '-1', '--oneline'], check=True)

# The committed E1 configuration is the reference for every identity check below.
E1_CONFIG_PATH = os.path.join(PROJECT_ROOT, 'configs', 'experiments', 'exp_e1_dice_ce.yaml')
assert os.path.isfile(E1_CONFIG_PATH), f'E1 config not found: {E1_CONFIG_PATH}'
print('E1 config  :', E1_CONFIG_PATH)


In [ ]:
# Link the Drive dataset into the repository so the committed relative paths
# in the E1 config resolve unchanged -- the same mechanism the training
# notebooks use.
DRIVE_DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')
assert os.path.isdir(DRIVE_DATASET_DIR), f'Dataset not found on Drive: {DRIVE_DATASET_DIR}'

link_path = os.path.join(PROJECT_ROOT, 'dataset')
if os.path.islink(link_path):
    os.remove(link_path)
elif os.path.exists(link_path):
    raise RuntimeError(f'{link_path} exists and is not a symlink -- refusing to overwrite.')
os.symlink(DRIVE_DATASET_DIR, link_path)

TRAIN_CSV = os.path.join(PROJECT_ROOT, 'dataset', 'prostate158_train', 'train.csv')
VALID_CSV = os.path.join(PROJECT_ROOT, 'dataset', 'prostate158_train', 'valid.csv')
for path in (TRAIN_CSV, VALID_CSV):
    assert os.path.isfile(path), f'Required split file missing: {path}'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Dataset linked :', link_path, '->', os.readlink(link_path))
print('Output dir     :', OUTPUT_DIR)


## 4. Environment and dependency check

Evaluation is FP32 on every device, so a GPU changes the runtime but not the
numbers. SciPy is required for the surface-distance metrics (HD95, ASD).


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nibabel', 'scipy', 'pandas', 'pyyaml'], check=True)

import platform, random
import nibabel, numpy as np, pandas as pd, scipy, torch, yaml

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None

# Deterministic evaluation. Inference is already deterministic (no dropout at
# eval time, no augmentation, no sampling), but the seed is set and recorded so
# the run is reproducible end to end.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ENVIRONMENT = {
    'python': sys.version.split()[0], 'platform': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': GPU_NAME,
    'numpy': np.__version__, 'scipy': scipy.__version__,
    'nibabel': nibabel.__version__, 'pandas': pd.__version__,
}
for key, value in ENVIRONMENT.items():
    print(f'  {key:<10}: {value}')
print(f'  {"device":<10}: {DEVICE}')
print(f'  {"seed":<10}: {SEED}')
if DEVICE == 'cpu':
    print('\n  NOTE: no GPU detected. Inference over 19 volumes will be slower,')
    print('        but the metrics are identical -- evaluation is FP32 everywhere.')


## 5. Checkpoint verification

The weights evaluated here must be bit-identical to the audited frozen candidate.
Architecture is verified by rebuilding the model from the **checkpoint's own
embedded configuration** and comparing it against a model rebuilt from the
**committed** E1 configuration — not by comparing a hard-coded parameter count.

Any failure stops the notebook before the test set is touched.


In [ ]:
import hashlib
from src.losses import LossConfig
from src.model import build_model
from src.train import verify_checkpoint_compatibility
from src.utils import load_config

assert os.path.isfile(CHECKPOINT_PATH), (
    f'Frozen E1 checkpoint not found: {CHECKPOINT_PATH}')

digest = hashlib.sha256()
with open(CHECKPOINT_PATH, 'rb') as f:
    for block in iter(lambda: f.read(1 << 20), b''):
        digest.update(block)
CHECKPOINT_SHA256 = digest.hexdigest()
CHECKPOINT_SIZE = os.path.getsize(CHECKPOINT_PATH)

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
checkpoint_config = checkpoint.get('config') or {}
committed_config = load_config(E1_CONFIG_PATH)
loss_spec = LossConfig.from_config(checkpoint_config)

model_from_checkpoint = build_model(checkpoint_config)
model_from_committed = build_model(committed_config)
n_params = sum(p.numel() for p in model_from_checkpoint.parameters())
n_params_committed = sum(p.numel() for p in model_from_committed.parameters())
architectures_match = (
    n_params == n_params_committed
    and [tuple(p.shape) for p in model_from_checkpoint.parameters()]
    == [tuple(p.shape) for p in model_from_committed.parameters()])
del model_from_committed

compat = verify_checkpoint_compatibility(checkpoint_config, committed_config, CHECKPOINT_PATH)

checks = []
def check(label, ok, detail=''):
    checks.append((label, bool(ok)))
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}{(' -- ' + str(detail)) if detail else ''}")

print('Checkpoint verification')
print('-' * 70)
print(f'  path   : {CHECKPOINT_PATH}')
print(f'  size   : {CHECKPOINT_SIZE:,} bytes')
print(f'  SHA-256: {CHECKPOINT_SHA256}')
print('-' * 70)
check('SHA-256 matches the frozen candidate', CHECKPOINT_SHA256 == EXPECTED_SHA256)
check('epoch is 88', checkpoint.get('epoch') == EXPECTED_EPOCH, checkpoint.get('epoch'))
check('experiment is exp_e1_dice_ce',
      checkpoint_config.get('experiment', {}).get('name') == EXPERIMENT_NAME,
      checkpoint_config.get('experiment', {}).get('name'))
check('loss is Dice + CE', loss_spec.name == EXPECTED_LOSS, loss_spec.name)
check('CE class weights are None (E1, not E3)',
      loss_spec.ce_class_weights is EXPECTED_CE_CLASS_WEIGHTS, loss_spec.ce_class_weights)
check('Dice classes are (1, 2)', loss_spec.dice_classes == (CG, PZ), loss_spec.dice_classes)
check('architecture matches the committed E1 config',
      architectures_match, f'{n_params:,} parameters')
check('weights are finite',
      all(torch.isfinite(v).all() for v in checkpoint['model_state_dict'].values()
          if v.is_floating_point()))
check('config compatible with the committed E1 config',
      compat['verified'] or compat['skipped_reason'] is not None, compat['mismatches'])
check('checkpoint declares no test split',
      'test_csv' not in checkpoint_config.get('data', {})
      and 'test_dir' not in checkpoint_config.get('data', {}))

failed = [label for label, ok in checks if not ok]
assert not failed, (
    'CHECKPOINT VERIFICATION FAILED -- this is not the audited frozen E1 '
    f'checkpoint: {failed}. Do not proceed.')
print('-' * 70)
print(f'Checkpoint verified: {len(checks)}/{len(checks)} checks passed.')


In [ ]:
# The preprocessing that will be applied to the test cases comes from the
# checkpoint's own embedded config, so it is identical to what trained the
# weights. Shown here explicitly, and confirmed equal to the committed config.
from src.transforms import PreprocessingConfig

PREPROCESSING = PreprocessingConfig.from_dict(checkpoint_config.get('preprocessing', {}))
preprocessing_committed = PreprocessingConfig.from_dict(committed_config.get('preprocessing', {}))

print('Preprocessing (from the checkpoint, identical to E1 training/validation)')
print('-' * 70)
for field in ('normalization', 'z_resample', 'spatial_mode', 'target_size',
              'target_spacing_z'):
    print(f'  {field:<18}: {getattr(PREPROCESSING, field)}')
assert PREPROCESSING == preprocessing_committed, (
    'Preprocessing in the checkpoint differs from the committed E1 config.')
print('-' * 70)
print('Matches the committed E1 configuration exactly.')


## 6. Test dataset audit

`scripts/audit_test_archive.py` inspects the archive **read-only**: it loads no
model, runs no inference and computes no segmentation metric. It reports case
count and IDs, per-case presence of `t2.nii.gz` and `t2_anatomy_reader1.nii.gz`,
NIfTI geometry and label values, and overlap against the training and validation
cohorts.

The archive is never downloaded automatically. If it is absent, the notebook stops
and tells you exactly where it was expected.


In [ ]:
import json

archive_present = os.path.isfile(TEST_ARCHIVE_PATH)
extracted_present = os.path.isdir(TEST_EXTRACT_DIR)
print(f'  archive   : {TEST_ARCHIVE_PATH}  ->  '
      f"{'found' if archive_present else 'not present'}")
print(f'  extracted : {TEST_EXTRACT_DIR}  ->  '
      f"{'found' if extracted_present else 'not present'}")

if not archive_present and not extracted_present:
    raise FileNotFoundError(
        'The official Prostate158 test archive was not found.\n'
        f'  expected ZIP       : {TEST_ARCHIVE_PATH}\n'
        f'  expected extracted : {TEST_EXTRACT_DIR}\n'
        'Obtain it from DOI 10.5281/zenodo.6592345 and place it under '
        '<DRIVE_ROOT>/dataset/test/, or adjust TEST_ARCHIVE_PATH / '
        'TEST_EXTRACT_DIR in section 2. Nothing is downloaded automatically.')

AUDIT_JSON = os.path.join(OUTPUT_DIR, 'test_archive_audit.json')
audit_cmd = [sys.executable, 'scripts/audit_test_archive.py',
             '--extract-dir', TEST_EXTRACT_DIR,
             '--train-csv', TRAIN_CSV,
             '--valid-csv', VALID_CSV,
             '--audit-json', AUDIT_JSON]
if archive_present and not extracted_present:
    audit_cmd += ['--zip', TEST_ARCHIVE_PATH]

returncode = subprocess.run(audit_cmd, cwd=PROJECT_ROOT).returncode
assert returncode == 0, 'Test archive audit failed -- see the output above.'

with open(AUDIT_JSON, encoding='utf-8') as f:
    audit = json.load(f)
TEST_CASE_ROOT = audit.get('case_root') or TEST_EXTRACT_DIR
TEST_CASE_IDS = audit.get('case_ids') or []
print(f'\nAudit written to: {AUDIT_JSON}')


## 7. Data split and leakage verification

The official Prostate158 split is **119 train / 20 validation / 19 test**. The test
cases must be disjoint from both other cohorts: a single shared case would mean the
model is being scored on data it saw during training or selection.

The checks below read the audit payload by its exact key names and assert the schema
first, so a missing key fails loudly instead of letting a gate pass vacuously.


In [ ]:
from src.splits import load_official_split

REQUIRED_KEYS = ('case_ids', 'num_with_t2', 'num_with_gt', 'overlap_with_train',
                 'overlap_with_val', 'loader_resolves_19_test_ids',
                 'unique_gt_label_sets')
missing_keys = [k for k in REQUIRED_KEYS if k not in audit]
assert not missing_keys, (
    f'Audit payload is missing {missing_keys}. The audit script schema changed; '
    'the checks below cannot be trusted until they are updated.')

official = load_official_split(train_csv=TRAIN_CSV, valid_csv=VALID_CSV)
observed_labels = sorted({int(v) for group in audit['unique_gt_label_sets'] for v in group})

gates = []
def gate(label, ok, detail=''):
    gates.append((label, bool(ok)))
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}{(' -- ' + str(detail)) if detail else ''}")

print('Split and leakage verification')
print('-' * 70)
gate(f'{N_TRAIN_CASES} training cases', len(official.train_ids) == N_TRAIN_CASES,
     len(official.train_ids))
gate(f'{N_VAL_CASES} validation cases', len(official.val_ids) == N_VAL_CASES,
     len(official.val_ids))
gate(f'{N_TEST_CASES} test cases', len(TEST_CASE_IDS) == N_TEST_CASES, len(TEST_CASE_IDS))
gate('no test/train overlap', audit['overlap_with_train'] == [],
     audit['overlap_with_train'] or 'none')
gate('no test/validation overlap', audit['overlap_with_val'] == [],
     audit['overlap_with_val'] or 'none')
gate('no train/validation overlap',
     not (set(official.train_ids) & set(official.val_ids)))
gate('ground truth present for every test case',
     audit['num_with_gt'] == N_TEST_CASES, f"{audit['num_with_gt']}/{N_TEST_CASES}")
gate('T2 image present for every test case',
     audit['num_with_t2'] == N_TEST_CASES, f"{audit['num_with_t2']}/{N_TEST_CASES}")
gate('ground-truth labels within {0, 1, 2}',
     bool(observed_labels) and set(observed_labels).issubset(set(LABEL_MAPPING)),
     observed_labels)
gate('loader resolves exactly 19 test IDs',
     bool(audit['loader_resolves_19_test_ids']), audit.get('loader_message'))

failed_gates = [label for label, ok in gates if not ok]
assert not failed_gates, f'SPLIT / LEAKAGE VERIFICATION FAILED: {failed_gates}'
print('-' * 70)
print(f'Split verified: {len(gates)}/{len(gates)} gates passed.')
print(f'Test cases: {", ".join(TEST_CASE_IDS)}')


## 8. Final safety gates

Two last checks before the test plan is printed.

### Output isolation

The final test writes **only** to `results/test_e1_epoch88/`. The E1 experiment
directory and every other historical result directory are read-only here, so the
training/validation record cannot be overwritten.

### Output state — NONE / PARTIAL / COMPLETE

| State | Meaning | Action |
|---|---|---|
| **NONE** | no valid output | evaluation runs (needs confirmation) |
| **PARTIAL** | files missing, unreadable, wrong case count, or wrong checkpoint identity | the previous attempt produced **no valid result**, so re-running is legitimate and the partial output is replaced |
| **COMPLETE** | all required files, readable, 19 cases, epoch 88, matching SHA | the evaluation is **skipped**; existing results are loaded for reporting |

COMPLETE is never inferred from a filename alone — the summary must parse and must
describe this exact evaluation. This makes **Run all** idempotent: a finished
evaluation is never repeated, and an interrupted one can simply be re-run.


In [ ]:
# ---- Output isolation ---------------------------------------------------
PROTECTED_DIRS = ['exp01_baseline', 'exp_e1_dice_ce', 'exp_e2_ce_aug',
                  'exp_e3_dice_ce_pzweight', 'exp04_reconstruction',
                  'exp04_rq2_reconstruction']
output_abs = os.path.abspath(OUTPUT_DIR)
for name in PROTECTED_DIRS:
    protected = os.path.abspath(os.path.join(DRIVE_ROOT, 'results', name))
    assert output_abs != protected and not output_abs.startswith(protected + os.sep), (
        f'OUTPUT_DIR must not be inside the historical experiment directory {name}. '
        f'Got {output_abs}.')
assert os.path.basename(output_abs) == 'test_e1_epoch88', (
    f'OUTPUT_DIR must be .../results/test_e1_epoch88, got {output_abs}')
print('Output isolation : OK --', output_abs)
print('Protected (read-only here):', ', '.join(PROTECTED_DIRS))


In [ ]:
# ---- Output state -------------------------------------------------------
REQUIRED_OUTPUTS = ('metrics.csv', 'per_case_metrics.csv', 'summary.json',
                    'test_config.json', 'checkpoint_info.json',
                    'evaluation_report.md')
SUMMARY_PATH = os.path.join(OUTPUT_DIR, 'summary.json')

present_outputs = [name for name in REQUIRED_OUTPUTS
                   if os.path.isfile(os.path.join(OUTPUT_DIR, name))
                   and os.path.getsize(os.path.join(OUTPUT_DIR, name)) > 0]

OUTPUT_STATE = 'NONE'
prior_summary = None
state_reason = ''
if present_outputs:
    OUTPUT_STATE = 'PARTIAL'
    state_reason = f'{len(present_outputs)}/{len(REQUIRED_OUTPUTS)} required files present'
    if len(present_outputs) == len(REQUIRED_OUTPUTS):
        try:
            with open(SUMMARY_PATH, encoding='utf-8') as f:
                prior_summary = json.load(f)
            prior_run = prior_summary.get('run', {})
            per_case_rows = pd.read_csv(os.path.join(OUTPUT_DIR, 'per_case_metrics.csv'))
            n_cases_recorded = per_case_rows['patient_id'].nunique()
            if (prior_run.get('experiment') == EXPERIMENT_NAME
                    and prior_run.get('epoch') == EXPECTED_EPOCH
                    and prior_run.get('checkpoint_sha256') == EXPECTED_SHA256
                    and prior_run.get('split') == 'test'
                    and prior_run.get('n_cases') == N_TEST_CASES
                    and n_cases_recorded == N_TEST_CASES
                    and prior_summary.get('metrics')):
                OUTPUT_STATE = 'COMPLETE'
            else:
                state_reason = (f"summary describes experiment={prior_run.get('experiment')!r}, "
                                f"epoch={prior_run.get('epoch')}, "
                                f"n_cases={prior_run.get('n_cases')} "
                                f'(per-case rows cover {n_cases_recorded} cases)')
        except (OSError, ValueError, KeyError) as exc:
            state_reason = f'existing output could not be read ({exc})'

print()
print('=' * 70)
print(f'OUTPUT STATE: {OUTPUT_STATE}')
print('=' * 70)
if OUTPUT_STATE == 'NONE':
    print('  No previous held-out test result. The evaluation will run.')
elif OUTPUT_STATE == 'PARTIAL':
    print(f'  An earlier attempt left incomplete output ({state_reason}).')
    print('  No valid result was produced, so re-running is legitimate.')
    print('  Existing files will be replaced:', ', '.join(present_outputs))
else:
    run_info = prior_summary['run']
    print('  A COMPLETE held-out test result already exists:')
    print(f"    experiment : {run_info.get('experiment')} epoch {run_info.get('epoch')}")
    print(f"    cases      : {run_info.get('n_cases')}")
    print(f"    generated  : {run_info.get('generated_utc')}")
    print('  The evaluation will be SKIPPED and these results reused for reporting.')
    print('  To genuinely redo it, move the directory aside by hand and record why.')
print('=' * 70)

INFERENCE_NEEDED = OUTPUT_STATE in ('NONE', 'PARTIAL')


## 9. One-time test confirmation

Every check has now passed. This cell prints the test plan and then stops, unless
you have explicitly confirmed the run.

**To proceed:** set `ONE_TIME_RUN_CONFIRMED = True` in section 2 and Run all again.

The held-out set is evaluated once per frozen model. The numbers that come out are
reported as they are — they are not used to reconsider the model, the checkpoint,
the preprocessing or the thresholds.


In [ ]:
print('=' * 70)
print('FINAL HELD-OUT TEST PLAN')
print('=' * 70)
print(f'  experiment     : {EXPERIMENT_NAME} (Dice + Cross Entropy)')
print(f'  checkpoint     : epoch {EXPECTED_EPOCH}')
print(f'  SHA-256        : {CHECKPOINT_SHA256}')
print(f'  test cases     : {len(TEST_CASE_IDS)} (official held-out split)')
print(f'  preprocessing  : identical to E1 training/validation')
print(f'  augmentation   : none      post-processing: none      TTA: none')
print(f'  label mapping  : ' + ', '.join(f'{k}={v}' for k, v in LABEL_MAPPING.items()))
print(f'  device / seed  : {DEVICE} / {SEED}')
print(f'  output         : {OUTPUT_DIR}')
print(f'  output state   : {OUTPUT_STATE}')
print('=' * 70)

if not INFERENCE_NEEDED:
    print('\nA complete result already exists -- sections 10-11 will reuse it.')
elif not ONE_TIME_RUN_CONFIRMED:
    raise RuntimeError(
        'STOP -- the held-out test was NOT run.\n\n'
        'Every verification above passed and the test set has not been touched.\n'
        'Review the plan, then set ONE_TIME_RUN_CONFIRMED = True in section 2 and '
        'Run all again to perform the evaluation.')
else:
    print('\nConfirmation given -- the evaluation will run in section 11.')


## 10. Model initialization

The frozen weights are loaded into the exact E1 architecture and put in evaluation
mode. A single forward pass confirms the model is usable before the test cases are
read.

The evaluation in section 11 calls `src/evaluate.py::run_evaluation`, which rebuilds
the model from the checkpoint's embedded config through this same code path. That is
deliberate: the test numbers are produced by exactly the code that produced the
validation numbers, so the two are directly comparable.


In [ ]:
if INFERENCE_NEEDED:
    model = model_from_checkpoint
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    model.to(DEVICE)
    model.eval()

    in_channels = checkpoint_config['model']['in_channels']
    height, width = PREPROCESSING.target_size
    with torch.no_grad():
        logits = model(torch.zeros(1, in_channels, height, width, device=DEVICE))

    assert tuple(logits.shape) == (1, len(LABEL_MAPPING), height, width), (
        f'unexpected output shape {tuple(logits.shape)}')
    assert torch.isfinite(logits).all(), 'model produced non-finite logits'

    print(f"  architecture : {checkpoint_config['model']['architecture']}")
    print(f'  channels     : {in_channels} -> {len(LABEL_MAPPING)}')
    print(f'  parameters   : {n_params:,}')
    print(f'  input size   : {height} x {width}')
    print(f'  mode         : eval (dropout off, no augmentation, no TTA)')
    print(f'  device       : {DEVICE}')
    print(f'  forward pass : OK {tuple(logits.shape)}')

    del logits
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
else:
    print('Model initialization skipped -- a complete result already exists.')


## 11. Held-out test inference

Inference over all 19 official test cases: `model.eval()` under `torch.no_grad()`,
FP32, argmax over three classes, per-case, at volume level, in original voxel space,
against the untouched on-disk ground truth.

The evaluation configuration is the committed E1 config with exactly two additions —
`data.test_dir` (required to reach the test split, and deliberately absent during
training) and `experiment.output_dir` (the isolated output directory). Nothing else
differs, and the cell asserts it.


In [ ]:
import copy, time
from src.evaluate import run_evaluation


def flatten(mapping, prefix=''):
    flat = {}
    for key, value in mapping.items():
        dotted = f'{prefix}{key}'
        if isinstance(value, dict):
            flat.update(flatten(value, dotted + '.'))
        else:
            flat[dotted] = value
    return flat


test_config = copy.deepcopy(committed_config)
test_config['experiment']['output_dir'] = OUTPUT_DIR
test_config['data']['test_dir'] = TEST_CASE_ROOT

ALLOWED_DEVIATIONS = {'experiment.output_dir', 'data.test_dir'}
deviations = {k for k in set(flatten(committed_config)) | set(flatten(test_config))
              if flatten(committed_config).get(k, '<absent>')
              != flatten(test_config).get(k, '<absent>')}
assert deviations == ALLOWED_DEVIATIONS, (
    f'Test config deviates from the committed E1 config in {sorted(deviations)}; '
    f'only {sorted(ALLOWED_DEVIATIONS)} are permitted.')

TEST_CONFIG_YAML = os.path.join(OUTPUT_DIR, 'test_config.yaml')
SESSION_CONFIG_YAML = '/content/e1_final_test_config.yaml'
for path in (TEST_CONFIG_YAML, SESSION_CONFIG_YAML):
    with open(path, 'w') as f:
        yaml.safe_dump(test_config, f, sort_keys=False, default_flow_style=False)

if INFERENCE_NEEDED:
    print('=' * 70)
    print(f'HELD-OUT TEST INFERENCE -- {EXPERIMENT_NAME} epoch {EXPECTED_EPOCH}')
    print('=' * 70)
    started = time.time()
    evaluation = run_evaluation(
        config_path=SESSION_CONFIG_YAML,
        checkpoint_path=CHECKPOINT_PATH,
        output_dir=OUTPUT_DIR,
        split='test',
        device=DEVICE,
        class_ids=tuple(LABEL_MAPPING),
    )
    ELAPSED_SECONDS = time.time() - started
    summary_metrics = evaluation['summary']
    metadata = evaluation['metadata']
    print(f'\nElapsed: {ELAPSED_SECONDS:.1f} s')
else:
    ELAPSED_SECONDS = None
    metadata = prior_summary['run']
    summary_metrics = None   # rebuilt from the stored CSV in section 13
    print('Inference skipped -- reusing the existing complete result.')

if INFERENCE_NEEDED:
    assert metadata['n_cases'] == N_TEST_CASES, (
        f"evaluated {metadata['n_cases']} cases, expected {N_TEST_CASES}")
    assert metadata['is_official_held_out_test'] is True
    assert metadata['checkpoint_epoch'] == EXPECTED_EPOCH
    print('\nHeld-out test evaluation complete.')


## 12. Per-case metrics

Dice, IoU, HD95, ASD, precision and recall for every test case and every class,
computed at volume level in original voxel space. This table is the raw record the
aggregates in section 13 are derived from.


In [ ]:
PER_CASE_PATH = os.path.join(OUTPUT_DIR, 'per_case_metrics.csv')

if INFERENCE_NEEDED:
    # run_evaluation writes eval_test_*; rename to this notebook's canonical
    # filenames so the output directory contains one copy of each artifact.
    os.replace(evaluation['per_case_csv'], PER_CASE_PATH)
    os.replace(evaluation['summary_csv'], os.path.join(OUTPUT_DIR, 'metrics.csv'))
    os.replace(evaluation['summary_json'],
               os.path.join(OUTPUT_DIR, 'evaluation_summary_raw.json'))

per_case = pd.read_csv(PER_CASE_PATH)
METRIC_NAMES = ('dice', 'iou', 'hd95_mm', 'asd_mm', 'precision', 'recall')

assert per_case['patient_id'].nunique() == N_TEST_CASES, (
    f"per-case table covers {per_case['patient_id'].nunique()} cases, "
    f'expected {N_TEST_CASES}')
assert not per_case[list(METRIC_NAMES)].isna().any().any(), (
    'per-case metrics contain missing values')

print(f'Per-case metrics: {len(per_case)} rows, '
      f"{per_case['patient_id'].nunique()} cases, "
      f"{per_case['class_id'].nunique()} classes")
print()
print(per_case.to_string(index=False))


In [ ]:
# Per-case macro Dice, defined exactly as in the E1 validation evaluation:
# the mean of the two FOREGROUND classes. Background is excluded.
foreground = per_case[per_case['class_id'].isin([CG, PZ])]
macro_per_case = foreground.pivot(index='patient_id', columns='class_id', values='dice')
macro_per_case['macro_dice'] = macro_per_case[[CG, PZ]].mean(axis=1)
macro_per_case = macro_per_case.rename(columns={CG: 'cg_dice', PZ: 'pz_dice'})

print('Per-case macro Dice   (macro = mean of CG and PZ; background excluded)')
print('-' * 70)
print(macro_per_case.to_string())
print('-' * 70)
print(f"mean {macro_per_case['macro_dice'].mean():.4f} "
      f"+/- {macro_per_case['macro_dice'].std(ddof=0):.4f} SD   |   "
      f"median {macro_per_case['macro_dice'].median():.4f}   |   "
      f"range {macro_per_case['macro_dice'].min():.4f}-{macro_per_case['macro_dice'].max():.4f}")


## 13. Aggregate metrics

Cohort means with standard deviation across the 19 test cases. Macro Dice is the
mean of CG and PZ, matching the E1 validation definition — background is reported
separately and is **not** part of macro Dice.


In [ ]:
# ddof=0 (population SD), matching src/metrics.py -- which is what produced the
# validation, baseline, E2 and E3 standard deviations. Pandas defaults to ddof=1;
# using that here would put two SD conventions in one results table and make the
# validation-vs-test comparison inconsistent.
aggregate = (per_case.groupby('class_id')[list(METRIC_NAMES)]
             .agg(['mean', lambda x: x.std(ddof=0)]))
aggregate.columns = aggregate.columns.set_levels(['mean', 'std'], level=1)

TEST_CG_DICE = float(aggregate.loc[CG, ('dice', 'mean')])
TEST_PZ_DICE = float(aggregate.loc[PZ, ('dice', 'mean')])
TEST_MACRO_DICE = (TEST_CG_DICE + TEST_PZ_DICE) / 2

print('=' * 78)
print(f'E1 HELD-OUT TEST METRICS -- {N_TEST_CASES} official test cases')
print('per-case, volume-level, original voxel space')
print('=' * 78)
print(f"  {'class':<12}{'metric':<12}{'mean':>12}{'SD':>12}")
print('-' * 78)
for class_id, class_name in LABEL_MAPPING.items():
    for metric in METRIC_NAMES:
        mean = aggregate.loc[class_id, (metric, 'mean')]
        std = aggregate.loc[class_id, (metric, 'std')]
        print(f'  {class_name:<12}{metric:<12}{mean:>12.4f}{std:>12.4f}')
    print('-' * 78)
print(f'  MACRO DICE (CG + PZ) / 2 : {TEST_MACRO_DICE:.4f}')
print('=' * 78)


In [ ]:
# Validation (used for selection, optimistically biased) vs test (unbiased).
comparison_rows = [
    ('CG Dice',      E1_VALIDATION['cg_dice'],      TEST_CG_DICE, True),
    ('PZ Dice',      E1_VALIDATION['pz_dice'],      TEST_PZ_DICE, True),
    ('Macro Dice',   E1_VALIDATION['macro_dice'],   TEST_MACRO_DICE, True),
    ('CG precision', E1_VALIDATION['cg_precision'], float(aggregate.loc[CG, ('precision', 'mean')]), True),
    ('CG recall',    E1_VALIDATION['cg_recall'],    float(aggregate.loc[CG, ('recall', 'mean')]), True),
    ('PZ precision', E1_VALIDATION['pz_precision'], float(aggregate.loc[PZ, ('precision', 'mean')]), True),
    ('PZ recall',    E1_VALIDATION['pz_recall'],    float(aggregate.loc[PZ, ('recall', 'mean')]), True),
    ('CG HD95 (mm)', E1_VALIDATION['cg_hd95'],      float(aggregate.loc[CG, ('hd95_mm', 'mean')]), False),
    ('PZ HD95 (mm)', E1_VALIDATION['pz_hd95'],      float(aggregate.loc[PZ, ('hd95_mm', 'mean')]), False),
    ('CG ASD (mm)',  E1_VALIDATION['cg_asd'],       float(aggregate.loc[CG, ('asd_mm', 'mean')]), False),
    ('PZ ASD (mm)',  E1_VALIDATION['pz_asd'],       float(aggregate.loc[PZ, ('asd_mm', 'mean')]), False),
]

print('=' * 78)
print('VALIDATION (20 cases, used for selection) vs TEST (19 cases, held out)')
print('=' * 78)
print(f"  {'metric':<15}{'validation':>12}{'test':>12}{'difference':>13}  direction")
print('-' * 78)
for name, validation_value, test_value, higher_is_better in comparison_rows:
    direction = 'higher is better' if higher_is_better else 'lower is better'
    print(f'  {name:<15}{validation_value:>12.4f}{test_value:>12.4f}'
          f'{test_value - validation_value:>+13.4f}  {direction}')
print('=' * 78)
print('The validation column drove model selection and is therefore optimistically')
print('biased. The test column is the unbiased estimate and is what the thesis')
print('reports. A gap between them is expected, not a defect.')


## 14. Save final test results to Google Drive

Everything is written under `results/test_e1_epoch88/` on Drive, so the results
survive the Colab session and no historical experiment directory is touched.

| File | Contents |
|---|---|
| `per_case_metrics.csv` | one row per case per class, all six metrics |
| `metrics.csv` | cohort aggregate per class per metric |
| `summary.json` | run identity, aggregate metrics, macro Dice, validation comparison |
| `test_config.json` | configuration, preprocessing, dataset, label mapping, seed |
| `checkpoint_info.json` | checkpoint path, SHA-256, epoch, architecture, loss |
| `evaluation_report.md` | human-readable report for the thesis |
| `test_config.yaml` | the resolved config passed to the evaluator |
| `test_archive_audit.json` | the read-only archive audit |
| `evaluation_summary_raw.json` | the evaluator's own summary, with bootstrap CIs |


In [ ]:
import datetime

GENERATED_UTC = datetime.datetime.now(datetime.timezone.utc).isoformat().replace('+00:00', 'Z')

checkpoint_info = {
    'experiment': EXPERIMENT_NAME,
    'checkpoint_path': CHECKPOINT_PATH,
    'checkpoint_sha256': CHECKPOINT_SHA256,
    'checkpoint_size_bytes': CHECKPOINT_SIZE,
    'epoch': int(checkpoint.get('epoch')),
    'selection': {
        'split_used': 'validation (20 official cases)',
        'rule': 'highest validation macro Dice (pre-committed)',
        'validation_macro_dice': E1_VALIDATION['macro_dice'],
        'test_metrics_used_for_selection': False,
    },
    'model': checkpoint_config.get('model'),
    'parameters': int(n_params),
    'loss': loss_spec.to_dict(),
    'architecture_matches_committed_config': bool(architectures_match),
    'verified_utc': GENERATED_UTC,
}

test_config_record = {
    'experiment': EXPERIMENT_NAME,
    'evaluation': 'official held-out test',
    'generated_utc': GENERATED_UTC,
    'git_commit': GIT_COMMIT,
    'repository': REPO_URL,
    'notebook': 'notebooks/prostate158_e1_final_test_colab.ipynb',
    'config_committed': 'configs/experiments/exp_e1_dice_ce.yaml',
    'dataset': {
        'name': 'Prostate158',
        'test_case_root': TEST_CASE_ROOT,
        'n_test_cases': int(N_TEST_CASES),
        'test_case_ids': list(TEST_CASE_IDS),
        'official_split': {'train': N_TRAIN_CASES, 'validation': N_VAL_CASES,
                           'test': N_TEST_CASES},
        'target_mask': committed_config['data']['target_mask'],
    },
    'preprocessing': {
        'normalization': PREPROCESSING.normalization,
        'z_resample': PREPROCESSING.z_resample,
        'target_spacing_z': PREPROCESSING.target_spacing_z,
        'spatial_mode': PREPROCESSING.spatial_mode,
        'target_size': list(PREPROCESSING.target_size),
        'source': "checkpoint's embedded config (identical to E1 training)",
    },
    'label_mapping': {str(k): v for k, v in LABEL_MAPPING.items()},
    'label_mapping_status': 'verified_multiplanar_2026_10_05',
    'inference': {
        'mode': 'eval', 'grad': 'disabled', 'precision': 'FP32',
        'augmentation': 'none', 'test_time_augmentation': 'none',
        'post_processing': 'none', 'seed': SEED, 'device': DEVICE,
        'elapsed_seconds': ELAPSED_SECONDS,
    },
    'environment': ENVIRONMENT,
    'test_set_usage_history': [
        {'date': '2026-09-13', 'what': 'baseline test evaluation',
         'checkpoint': 'exp01_baseline epoch 77'},
        {'date': '2026-09-19', 'what': 'RQ2 reconstruction geometry study',
         'checkpoint': 'exp01_baseline epoch 77'},
        {'date': GENERATED_UTC[:10], 'what': 'E1 held-out test evaluation (this run)',
         'checkpoint': f'{EXPERIMENT_NAME} epoch {EXPECTED_EPOCH}'},
    ],
}

metrics_record = {
    class_name: {
        metric: {'mean': float(aggregate.loc[class_id, (metric, 'mean')]),
                 'std': float(aggregate.loc[class_id, (metric, 'std')])}
        for metric in METRIC_NAMES
    }
    for class_id, class_name in LABEL_MAPPING.items()
}

summary_record = {
    'run': {
        'experiment': EXPERIMENT_NAME,
        'epoch': EXPECTED_EPOCH,
        'checkpoint_sha256': CHECKPOINT_SHA256,
        'split': 'test',
        'n_cases': int(N_TEST_CASES),
        'is_official_held_out_test': True,
        'generated_utc': GENERATED_UTC,
        'git_commit': GIT_COMMIT,
    },
    'metrics': metrics_record,
    'macro_dice': {
        'definition': 'mean of CG and PZ Dice; background excluded',
        'test': TEST_MACRO_DICE,
        'validation': E1_VALIDATION['macro_dice'],
        'difference': TEST_MACRO_DICE - E1_VALIDATION['macro_dice'],
        'per_case_mean': float(macro_per_case['macro_dice'].mean()),
        'per_case_std': float(macro_per_case['macro_dice'].std(ddof=0)),
        'per_case_median': float(macro_per_case['macro_dice'].median()),
    },
    'validation_vs_test': {
        name: {'validation': validation_value, 'test': test_value,
               'difference': test_value - validation_value,
               'higher_is_better': higher_is_better}
        for name, validation_value, test_value, higher_is_better in comparison_rows
    },
    'label_mapping': {str(k): v for k, v in LABEL_MAPPING.items()},
    'scope_note': (
        'RQ1 only. No reconstruction, mesh generation or RQ2 evaluation is '
        'performed in this notebook.'
    ),
}

for filename, payload in (('checkpoint_info.json', checkpoint_info),
                          ('test_config.json', test_config_record),
                          ('summary.json', summary_record)):
    with open(os.path.join(OUTPUT_DIR, filename), 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, default=str)
    print('wrote', filename)


In [ ]:
report_lines = [
    '# E1 — Final Held-Out Test Evaluation (RQ1)',
    '',
    f'Generated {GENERATED_UTC} from commit `{GIT_COMMIT}`.',
    '',
    '## 1. Run identity',
    '',
    '| Field | Value |',
    '|---|---|',
    f'| Experiment | `{EXPERIMENT_NAME}` (Dice + Cross Entropy) |',
    f'| Checkpoint epoch | {EXPECTED_EPOCH} |',
    f'| Checkpoint SHA-256 | `{CHECKPOINT_SHA256}` |',
    f'| Split | official held-out test, {N_TEST_CASES} cases |',
    f'| Evaluation | per-case, volume-level, original voxel space |',
    '| Augmentation / TTA / post-processing | none |',
    f'| Device / seed | {DEVICE} / {SEED} |',
    '| Label mapping | 0 = Background, 1 = CG, 2 = PZ (verified 2026-10-05) |',
    '',
    '> E1 was selected on validation data only. These test numbers are reported as',
    '> measured and were not used to reconsider the model, checkpoint or preprocessing.',
    '',
    '## 2. Test results',
    '',
    '| Class | Dice | IoU | HD95 (mm) | ASD (mm) | Precision | Recall |',
    '|---|---|---|---|---|---|---|',
]
for class_id, class_name in LABEL_MAPPING.items():
    values = ' | '.join(
        f"{aggregate.loc[class_id, (m, 'mean')]:.4f} ± {aggregate.loc[class_id, (m, 'std')]:.4f}"
        for m in METRIC_NAMES)
    report_lines.append(f'| {class_name} | {values} |')
report_lines += [
    '',
    f'**Macro Dice (CG + PZ) / 2 = {TEST_MACRO_DICE:.4f}**  '
    f"(per-case {macro_per_case['macro_dice'].mean():.4f} ± "
    f"{macro_per_case['macro_dice'].std(ddof=0):.4f} SD, "
    f"median {macro_per_case['macro_dice'].median():.4f})",
    '',
    'Background is reported above for completeness and is **not** part of macro Dice.',
    '',
    '## 3. Validation vs test',
    '',
    '| Metric | Validation (20 cases) | Test (19 cases) | Difference |',
    '|---|---|---|---|',
]
for name, validation_value, test_value, _ in comparison_rows:
    report_lines.append(
        f'| {name} | {validation_value:.4f} | {test_value:.4f} | {test_value - validation_value:+.4f} |')
report_lines += [
    '',
    'Validation drove model selection and is optimistically biased; the test column',
    'is the unbiased estimate and is what the thesis reports.',
    '',
    '## 4. Per-case results',
    '',
    'See `per_case_metrics.csv` for all cases and classes. Per-case macro Dice:',
    '',
    '| Case | CG Dice | PZ Dice | Macro Dice |',
    '|---|---|---|---|',
]
for case_id, row in macro_per_case.iterrows():
    report_lines.append(
        f"| {case_id} | {row['cg_dice']:.4f} | {row['pz_dice']:.4f} | {row['macro_dice']:.4f} |")
report_lines += [
    '',
    '## 5. Provenance and scope',
    '',
    '* Test cases: ' + ', '.join(TEST_CASE_IDS),
    '* Preprocessing is the checkpoint\'s own embedded configuration, identical to',
    '  E1 training and validation.',
    '* The official test set was evaluated once for this model. **Historical note:**',
    '  the test set had previously been evaluated with the baseline experiment',
    '  (2026-09-13) and used for the RQ2 reconstruction geometry study (2026-09-19),',
    '  both with the Baseline epoch-77 checkpoint and both before the E1/E2/E3 arms',
    '  were designed. This E1 run is therefore not the first historical use of the',
    '  official test set.',
    '* Historical experiment directories were not modified by this evaluation.',
    '* **RQ1 only.** RQ2 reconstruction will use the frozen E1 model and is handled',
    '  in a separate reconstruction workflow.',
    '',
]

REPORT_PATH = os.path.join(OUTPUT_DIR, 'evaluation_report.md')
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines) + '\n')
print('wrote evaluation_report.md')


In [ ]:
# Verify every required artifact exists on Drive and is readable.
EXPECTED_ARTIFACTS = REQUIRED_OUTPUTS + ('test_config.yaml', 'test_archive_audit.json',
                                         'evaluation_summary_raw.json')

print(f"{'artifact':<30}{'size':>12}  status")
print('-' * 70)
missing_artifacts = []
for name in EXPECTED_ARTIFACTS:
    path = os.path.join(OUTPUT_DIR, name)
    exists = os.path.isfile(path)
    size = os.path.getsize(path) if exists else 0
    readable = False
    if exists and size > 0:
        try:
            with open(path, 'rb') as f:
                f.read(1)
            readable = True
        except OSError:
            readable = False
    if not readable:
        missing_artifacts.append(name)
    print(f"{name:<30}{size:>12,}  {'OK' if readable else 'MISSING/UNREADABLE'}")
print('-' * 70)
assert not missing_artifacts, f'Missing or unreadable artifacts: {missing_artifacts}'

# Historical experiment directories must be untouched by this notebook.
for name in PROTECTED_DIRS:
    path = os.path.join(DRIVE_ROOT, 'results', name)
    if os.path.isdir(path):
        print(f'{name:<30} present, not written to by this notebook')

print(f'\nAll results saved to: {OUTPUT_DIR}')


## 15. Final test summary


In [ ]:
print('=' * 78)
print('E1 HELD-OUT TEST — COMPLETE')
print('=' * 78)
print()
print(f'Checkpoint    : {EXPERIMENT_NAME} — epoch {EXPECTED_EPOCH}')
print(f'SHA-256       : {CHECKPOINT_SHA256}')
print(f'Test cases    : {N_TEST_CASES}')
print(f'Selection     : validation data only (20 cases), highest macro Dice')
print()
for class_id, class_name in ((CG, 'Central Gland (CG)'), (PZ, 'Peripheral Zone (PZ)')):
    print(f'{class_name}:')
    for metric in METRIC_NAMES:
        mean = aggregate.loc[class_id, (metric, 'mean')]
        std = aggregate.loc[class_id, (metric, 'std')]
        print(f'    {metric:<12} = {mean:.4f} ± {std:.4f}')
    print()
print(f'Macro Dice    : {TEST_MACRO_DICE:.4f}')
print(f'              (validation was {E1_VALIDATION["macro_dice"]:.4f}, '
      f'difference {TEST_MACRO_DICE - E1_VALIDATION["macro_dice"]:+.4f})')
print()
print(f'Output        : {OUTPUT_DIR}')
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(f'                  {name}')
print('=' * 78)
print()
print('RQ1 is now answered on the official held-out test set. These numbers are')
print('final: the model, checkpoint, preprocessing and thresholds are not revisited')
print('in light of them.')


## 16. RQ1 / RQ2 handoff

RQ1 is complete. The frozen E1 model (epoch 88, SHA-256
`2366e3c6…28d5d2`) and the results in `results/test_e1_epoch88/` are the
RQ1 deliverable.

> **RQ2 reconstruction will use the frozen E1 test predictions and will be handled
> in a separate reconstruction workflow.**

### What RQ2 will need

1. The same frozen checkpoint — verified by the SHA-256 above, not by filename.
2. The checkpoint references in the RQ2 workflow re-pointed from the Baseline
   epoch-77 weights to this E1 checkpoint:
   * `configs/experiments/exp04_rq2_reconstruction.yaml` → `source_checkpoint`
   * `notebooks/prostate158_rq2_reconstruction_colab.ipynb` → `CHECKPOINT_PATH`
3. A fresh RQ2 run. The existing `results/exp04_reconstruction/` and
   `results/exp04_rq2_reconstruction/` artifacts were produced with the Baseline
   checkpoint and must not be presented as E1 results.

### One reporting rule to carry into RQ2

Reconstruction fidelity and segmentation accuracy are different quantities and must
never share a table. The round-trip Dice of 1.0 proves the 2D→3D transform is
lossless — it involves no model and is **not** a segmentation result. The RQ2
headline numbers are the model's 3D segmentation metrics.

### Not performed in this notebook

3D reconstruction · mesh generation · surface reconstruction · RQ2 evaluation ·
RQ2 visualization.
